In [1]:
import sys; sys.path.append("..")
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from src.features import buat_fitur

d = pd.read_csv("../data/clean.csv", parse_dates=["waktu"], index_col="waktu")
d["lag_168"] = d["konsumsi_filled"].shift(168)

TRAIN_END = "2026-02-28 23:00"
VAL_START, VAL_END = "2026-03-01 00:00", "2026-04-30 23:00"
# test (Mei-Juni 2026) TIDAK disentuh di notebook ini

# hari "sulit": status libur berbeda dibanding 7 hari sebelumnya
beda_libur = d["hari_libur"] != d["hari_libur"].shift(168)

def laporan(nama, pred, y, mask):
    e = (pred - y)[mask]
    bl = beda_libur[mask]
    return {"model": nama, "n": int(mask.sum()),
            "MAE": e.abs().mean(),
            "MAE_minggu_biasa": e[~bl].abs().mean(),
            "MAE_minggu_libur": e[bl].abs().mean(),
            "bias": e.mean()}

y = d["konsumsi_kWh"]
in_val = (d.index >= VAL_START) & (d.index <= VAL_END)
rows = []

# baseline
mv = d["target_valid"] & d["lag_168"].notna() & in_val
rows.append(laporan("Baseline lag-168", d["lag_168"], y, mv))

# model A (tanpa lag) dan B (dengan lag)
for nama, pakai_lag in [("HGB A: tanpa lag", False), ("HGB B: dengan lag", True)]:
    X = buat_fitur(d, pakai_lag)
    ok = d["target_valid"] & X.notna().all(axis=1)
    tr = ok & (d.index <= TRAIN_END)
    va = ok & in_val
    m = HistGradientBoostingRegressor(loss="absolute_error", max_iter=400,
                                      learning_rate=0.05, early_stopping=False,
                                      random_state=0)
    m.fit(X[tr], y[tr])
    pred = pd.Series(m.predict(X), index=X.index)
    rows.append(laporan(nama, pred, y, va))

pd.DataFrame(rows).round(2)

,model,n,MAE,MAE_minggu_biasa,MAE_minggu_libur,bias
0,Baseline lag-168,1449,22.12,12.34,72.63,-0.33
1,HGB A: tanpa lag,1449,5.03,5.03,5.08,0.02
2,HGB B: dengan lag,1449,5.07,5.04,5.22,0.02
